In [2]:
import time
import pandas as pd
import sys; sys.path.append("..")
from src.config import RESULTS_DIR, MODEL_NAME, RANDOM_SEED
from src.eval.fairness import choose_threshold, group_metrics, fairness_gaps, bootstrap_ci

In [3]:
run = f"{MODEL_NAME}_seed{RANDOM_SEED}"
pred_dir = RESULTS_DIR / "predictions"

In [4]:

val  = pd.read_csv(pred_dir / f"{run}_fitz_val.csv")
test = pd.read_csv(pred_dir / f"{run}_fitz_test.csv")
ddi  = pd.read_csv(pred_dir / f"{run}_ddi.csv")

print(val.shape, test.shape, ddi.shape)

(648, 5) (648, 5) (656, 5)


In [5]:
THRESHOLD = choose_threshold(val, target_sensitivity=0.90)
print(f"frozen threshold: {THRESHOLD:.4f}")

frozen threshold: 0.0937


In [6]:
for name, df in [("fitz_val", val), ("fitz_test", test), ("ddi", ddi)]:
    all_auc = group_metrics(df, THRESHOLD).loc["all", "auc"]
    print(f"{name:10s} all AUC = {all_auc:.4f}")

fitz_val   all AUC = 0.8949
fitz_test  all AUC = 0.8716
ddi        all AUC = 0.6680


In [7]:
for name, df in [("fitz_test", test), ("ddi", ddi)]:
    print(f"\n=== {name} (threshold={THRESHOLD:.3f}) ===")
    print(group_metrics(df, THRESHOLD).round(3))


=== fitz_test (threshold=0.094) ===
              n  n_malignant    auc  sensitivity  specificity
skin_group                                                   
light       346          179  0.835        0.894        0.599
medium      240          114  0.929        0.930        0.802
dark         62           31  0.824        0.774        0.677
all         648          324  0.872        0.895        0.685

=== ddi (threshold=0.094) ===
              n  n_malignant    auc  sensitivity  specificity
skin_group                                                   
light       208           49  0.681        0.878        0.365
medium      241           74  0.665        0.838        0.449
dark        207           48  0.658        0.708        0.497
all         656          171  0.668        0.813        0.437


In [8]:
for name, df in [("fitz_test", test), ("ddi", ddi)]:
    g = fairness_gaps(df, THRESHOLD)
    print(f"\n{name}:")
    for k, v in g.items():
        print(f"  {k:38s} {v:+.4f}")


fitz_test:
  auc_gap_light_minus_dark               +0.0106
  sensitivity_gap_light_minus_dark       +0.1197
  equalized_odds_difference              +0.2028

ddi:
  auc_gap_light_minus_dark               +0.0224
  sensitivity_gap_light_minus_dark       +0.1692
  equalized_odds_difference              +0.1692


In [9]:
from fairlearn.metrics import equalized_odds_difference

for name, df in [("fitz_test", test), ("ddi", ddi)]:
    ours = fairness_gaps(df, THRESHOLD)["equalized_odds_difference"]
    theirs = equalized_odds_difference(
        df["target"], (df["prob"] >= THRESHOLD).astype(int),
        sensitive_features=df["skin_group"],
    )
    print(f"{name:10s} ours={ours:.4f}  fairlearn={theirs:.4f}  "
          f"diff={abs(ours - theirs):.2e}")

fitz_test  ours=0.2028  fairlearn=0.2028  diff=8.33e-17
ddi        ours=0.1692  fairlearn=0.1692  diff=0.00e+00


In [10]:
results = {}
for name, df in [("fitz_test", test), ("ddi", ddi)]:
    t0 = time.time()
    ci = bootstrap_ci(df, THRESHOLD, n_boot=2000, seed=42)
    results[name] = ci
    print(f"\n=== {name} — {time.time()-t0:.1f}s ===")
    print(ci.round(4))


=== fitz_test — 11.4s ===
                                   point   ci_lo   ci_hi  n_valid
quantity                                                         
light_auc                         0.8347  0.7913  0.8770     2000
light_sensitivity                 0.8939  0.8436  0.9385     2000
medium_auc                        0.9287  0.8954  0.9600     2000
medium_sensitivity                0.9298  0.8772  0.9737     2000
dark_auc                          0.8241  0.7076  0.9157     2000
dark_sensitivity                  0.7742  0.6129  0.9032     2000
auc_gap_light_minus_dark          0.0106 -0.0919  0.1341     2000
sensitivity_gap_light_minus_dark  0.1197 -0.0348  0.2822     2000
equalized_odds_difference         0.2028  0.1409  0.3490     2000

=== ddi — 11.7s ===
                                   point   ci_lo   ci_hi  n_valid
quantity                                                         
light_auc                         0.6808  0.5972  0.7584     2000
light_sensitivity           

In [11]:
for name in ["fitz_test", "ddi"]:
    ci = results[name]
    print(f"\n=== {name} ===")
    for q in ["light_sensitivity", "dark_sensitivity",
              "sensitivity_gap_light_minus_dark",
              "auc_gap_light_minus_dark"]:
        r = ci.loc[q]
        # does the gap interval exclude zero?
        note = ""
        if "gap" in q:
            if r["ci_lo"] > 0 or r["ci_hi"] < 0:
                note = "  ← excludes zero"
            else:
                note = "  ← includes zero (can't rule out luck)"
        print(f"  {q:38s} {r['point']:+.3f} "
              f"[{r['ci_lo']:+.3f}, {r['ci_hi']:+.3f}]{note}")


=== fitz_test ===
  light_sensitivity                      +0.894 [+0.844, +0.939]
  dark_sensitivity                       +0.774 [+0.613, +0.903]
  sensitivity_gap_light_minus_dark       +0.120 [-0.035, +0.282]  ← includes zero (can't rule out luck)
  auc_gap_light_minus_dark               +0.011 [-0.092, +0.134]  ← includes zero (can't rule out luck)

=== ddi ===
  light_sensitivity                      +0.878 [+0.776, +0.959]
  dark_sensitivity                       +0.708 [+0.562, +0.833]
  sensitivity_gap_light_minus_dark       +0.169 [+0.023, +0.335]  ← excludes zero
  auc_gap_light_minus_dark               +0.022 [-0.098, +0.144]  ← includes zero (can't rule out luck)


In [12]:
for name in ["fitz_test", "ddi"]:
    n = results[name]["n_valid"].iloc[0]
    print(f"{name}: n_valid={n} rounds (expect ~2000)")

fitz_test: n_valid=2000 rounds (expect ~2000)
ddi: n_valid=2000 rounds (expect ~2000)


In [13]:
for name, ci in results.items():
    path = RESULTS_DIR / f"{run}_{name}_bootstrap_ci.csv"
    ci.to_csv(path)
    print(f"saved -> {path.name}")

saved -> efficientnet_b0_seed42_fitz_test_bootstrap_ci.csv
saved -> efficientnet_b0_seed42_ddi_bootstrap_ci.csv
